<a href="https://colab.research.google.com/github/edwardoughton/spatial_computing/blob/main/08_02_Scripting_Spatial_Processing_via_GeoPandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8: Spatial processing with GeoPandas — Part 2

Today we build on Part 1 using **intersection and difference** around Charlottesville, Virginia. Our four study regions are Albemarle County, Charlottesville independent city, Greene County, and Fluvanna County.

**Question:** How much of each region lies inside or outside two proposed service zones?

We reuse the Virginia administrative polygons from earlier weeks and create two fictional service zones around illustrative points near Charlottesville and Crozet. These are teaching examples, not actual facilities or measured service areas. A circular buffer represents straight line distance, not travel time or access along roads.

Run examples in order. Task cells are for your own code. If imports fail, run `%pip install geopandas matplotlib contextily` in a separate cell and retry.


In [ ]:
%pip install geopandas matplotlib contextily

## GeoSeries recap

A `GeoSeries` is a one dimensional array of geometries. Other attribute columns are ordinary Pandas Series. The points below are abstract Cartesian examples, so they have no geographic CRS.


In [ ]:
# Example: Creating a GeoPandas Series with Point geometries
import geopandas as gpd
from shapely.geometry import Point

# Creating a GeoPandas Series
points = gpd.GeoSeries([Point(0, 0), Point(1, 1), Point(2, 2)])
points

## GeoPandas GeoDataFrame

Remember that a `GeoPandas` `GeoDataFrame` is a two dimensional data structure (e.g., a matrix) which extends the `Pandas` `DataFrame`, by including a `GeoPandas` `Series` as one of its columns, e.g., the geometry column.

Spatial information for each row in the `GeoDataFrame` is stored in this geometry column. See the example below to remind yourself.

In [ ]:
# Example: Creating a GeoPandas GeoDataFrame with Point geometries
import geopandas as gpd
from shapely.geometry import Point

# Here we specify a dictionary structure, and then convert to GeoDataFrame
# The dict key will be the column name
# The list of values the row information
data = {
    'Name': ['Location A', 'Location B', 'Location C'],
    'Type': ['Type D', 'Type F', 'Type X'],
    'geometry': [Point(2, 3), Point(4, 5), Point(0, 9)]
    }
gdf = gpd.GeoDataFrame(data)

# Inspect our GeoDataFrame
gdf

## Load our Virginia boundaries

Use `data/virginia.zip` from the course repository when running locally. In Colab, the code downloads the same fixed course snapshot used in Week 6. No manual upload or extraction is needed.

This is a historical GADM course snapshot, not a current boundary release. Retain the [GADM attribution](https://gadm.org/) and check its terms before reuse outside class.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import matplotlib.pyplot as plt

data_url = 'https://raw.githubusercontent.com/edwardoughton/spatial_computing/93d2e72b57741f7cd192f9664d62bafb93c06739/data/virginia.zip'
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
data_path = data_dir / 'virginia.zip'
urlretrieve(data_url, data_path)

Select four administrative units by name and inspect their types. Charlottesville is an independent city, so it is a separate record from Albemarle County. We use EPSG:3968 (NAD83 / Virginia Lambert), a projected CRS with meter units, for buffers and area calculations.


In [ ]:
virginia = gpd.read_file(data_path)
study_names = ['Albemarle', 'Charlottesville', 'Greene', 'Fluvanna']
boundaries_source = virginia.loc[
    virginia['NAME_2'].isin(study_names),
    ['GID_2', 'NAME_2', 'TYPE_2', 'geometry'],
].copy()
boundaries_source


In [ ]:
analysis_crs = 'EPSG:3968'
boundaries = boundaries_source.to_crs(analysis_crs)
print('Analysis CRS:', boundaries.crs)

## Add a map background

As in Part 1, use Contextily to add Esri World Imagery beneath our geometries. Transparent polygon fills let us see the imagery. An internet connection is needed to load the tiles.

Run the installation cell below before drawing the maps. Import it as `ctx`, then call `ctx.add_basemap()` after plotting the geometries. Passing `crs=boundaries.crs` aligns the background with our polygons. We keep our analysis in EPSG:3968 for area and distance calculations.

We use `zoom=10` to show more detail around Charlottesville. Keep the imagery attribution visible when saving or sharing a map. If the tile service is unavailable, comment out the `ctx.add_basemap()` line to continue with the geometry map.


In [ ]:
import contextily as ctx

ax = boundaries.plot(
    facecolor='none', edgecolor='red', linewidth=1.4,
    figsize=(11, 9), zorder=2,
)
ctx.add_basemap(ax, crs=boundaries.crs, source=ctx.providers.Esri.WorldImagery, zoom=10)
ax.set_title('Charlottesville and the surrounding study regions', fontsize=14, pad=12)
ax.set_axis_off()
plt.show()


These four polygons define our study area. Print their names and identifiers before selecting a region. Region names are easier to read; `GID_2` provides the dataset identifier.


In [ ]:
print(boundaries[['GID_2', 'NAME_2', 'TYPE_2']].to_string(index=False))


## Create two proposed service zones

Start with illustrative longitude/latitude points near Charlottesville and Crozet. Project the points before constructing 25,000 meter (25 km) buffers. Preserve a `zone_id` so we can identify source zones after clipping.

The circles deliberately overlap. Adding their areas counts shared space twice; later we will dissolve them to measure unique coverage.


In [ ]:
# Create a gdf of hypothetical sites
sites = gpd.GeoDataFrame(
    {'zone_id': ['A', 'B'], 'site_name': ['Charlottesville example', 'Crozet example']},
    geometry=gpd.points_from_xy([-78.477, -78.701], [38.030, 38.069]),
    crs='EPSG:4326',
).to_crs(analysis_crs)
sites

In [ ]:
# Make service zones
radius_m = 25_000
zones = sites.copy()
zones['geometry'] = sites.geometry.buffer(radius_m)
zones.head()


Plot both proposed zones over the study boundaries. Which zone appears to cover more than one region? Predict what will happen to a zone crossing an administrative boundary when we intersect it with that boundary.


In [ ]:
ax = boundaries.plot(facecolor='none', edgecolor='red', figsize=(11, 9))
zones.plot(ax=ax, column='zone_id', alpha=0.35, legend=True)
sites.plot(ax=ax, color='black', markersize=20)
ctx.add_basemap(ax, crs=boundaries.crs, source=ctx.providers.Esri.WorldImagery, zoom=10)
ax.set_title('Proposed service zones around Charlottesville and Crozet')
ax.set_axis_off()
plt.show()


## Intersection

Intersection retains overlapping polygon portions and attributes from both layers.

Here, we are following the `GeoPandas` documentation page on set operations: https://geopandas.org/en/stable/docs/user_guide/set_operations.html

In this case, we will utilize the `.overlay()` function and specify "intersection".

Inspect and understand the difference below once this intersection has been carried out.  


In [ ]:
boundary = boundaries.loc[boundaries['NAME_2'] == 'Albemarle'].copy()
zones_inside = zones.overlay(boundary, how='intersection', keep_geom_type=True)
zones_inside['area_km2'] = zones_inside.geometry.area / 1e6
ax = boundary.plot(facecolor='none', edgecolor='black', figsize=(11, 9))
zones_inside.plot(ax=ax, column='zone_id', alpha=0.4, legend=True)
ctx.add_basemap(ax, crs=boundaries.crs, source=ctx.providers.Esri.WorldImagery, zoom=10)
ax.set_title('Proposed zones clipped to Albemarle County')
ax.set_axis_off()
plt.show()


## Task: intersect two regions

Select Albemarle and Charlottesville using `.isin()`, then intersect the original zones with that selection.

* Print region names, source zone IDs, output rows, and unique source zone count.
* Calculate each clipped record's area in square kilometers.
* Map the result with `ctx.add_basemap(ax, crs=boundaries.crs, source=ctx.providers.Esri.WorldImagery, zoom=10)` and export it to GeoPackage using `.to_file(..., driver='GPKG')`.
* Explain why a source zone can appear in multiple region records and why adding overlapping zone areas does not measure unique coverage.


In [ ]:
# Enter your attempt here


## Difference

Difference retains portions of the left layer outside the right layer. Subtract Albemarle from the zones, using the same selected boundary as above. Changing layer order changes the question being answered.

For this check, we sum each source zone separately. Inside area plus outside area should equal the sum of the original zone areas, even though overlapping zones count shared space twice in all three totals.


In [ ]:
import math

# Get difference
zones_outside = zones.overlay(boundary, how='difference', keep_geom_type=True)
original_area = zones.geometry.area.sum()
inside_area = zones_inside.geometry.area.sum()
outside_area = zones_outside.geometry.area.sum()
print('Original summed zone area (km2):', original_area / 1e6)
print('Inside + outside (km2):', (inside_area + outside_area) / 1e6)
print('Area residual (m2):', original_area - inside_area - outside_area)

In [ ]:
# Plot
ax = boundaries.plot(facecolor='none', edgecolor='gray', figsize=(11, 9))
zones_outside.plot(ax=ax, column='zone_id', alpha=0.4, legend=True)
boundary.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=2)
ctx.add_basemap(ax, crs=boundaries.crs, source=ctx.providers.Esri.WorldImagery, zoom=10)
ax.set_title('Zone portions outside Albemarle County')
ax.set_axis_off()
plt.show()